In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path

# Project imports
from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_gpr import Urc1_GPR
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess


In [2]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G6M2.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\plots\\GPR\\G6M2_comparison"
SAVE_PLOTS = True  # Set to False to disable saving plots

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.28,
        "Tref": 57,
        "OHref": 10,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 33,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.31,
        "Tref": 58,
        "OHref": 18,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv",
    },
}

# Model fitting common config (shared across all models)
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
    "data_filter_h_since_last_start_min": 0.5,
}

# Model variants to run
MODEL_VARIANTS = ["I2", "gpr"]  # Options: "I2", "cbrt", "sigmoid", "gpr"

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True

In [3]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)
preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name
print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} → {data.index.max()}\n")

# Preprocess once and reuse
shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G6M2_20260504_171005.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G6M2, shape: (1096020, 3)
Time range: 2023-07-06 00:00:00 → 2025-08-05 09:29:00

[preprocess_once] 1096020 -> 359353 points.
Shared preprocessed rows: 359353



In [4]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)
models = {}
for variant in MODEL_VARIANTS:
    print(f"\n  → Training {variant} model...")
    try:
        if variant == "gpr":
            model = Urc1_GPR(
                data=data,
                name=dataset_name,
                preprocessed_data=shared_pre,
                gpr_r2_threshold=0.8,
                gpr_cond_threshold=1e6,
                length_scale_init=500 * 24,
                output_mode="all",
                **COMMON_CONFIG,
            )
            models[f"GPR Smoothed"] = model
        else:
            model = Urc1(
                data=data,
                name=dataset_name,
                nonlinear_term=variant,
                preprocessed_data=shared_pre,
                **COMMON_CONFIG,
            )
            models[f"Baseline ({variant})"] = model
        print(f"  ✓ {variant} training completed")
    except Exception as e:
        print(f"  ✗ {variant} training failed: {e}")
print(f"\n✓ {len(models)} models trained successfully\n")

STEP 2: Model Training

  → Training I2 model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 218 intervals low data, 0 fit failed.
468 out of 762 fitting results are reliable.
  ✓ I2 training completed

  → Training gpr model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 218 intervals low data, 0 fit failed.
468 out of 762 fitting results are reliable.


c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 2400.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.2. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 2400.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_co

  ✓ gpr training completed

✓ 2 models trained successfully



c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.2. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)
comparator = UnifiedModelComparator(models)
print(f"✓ UnifiedModelComparator initialized with {len(models)} models\n")

# Load GT for each reference configuration
gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()
    print(f"Looking for GT file: {gt_path}")
    
    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break
            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(f"Cannot identify voltage column in {gt_path}. Columns: {gt_data.columns.tolist()}")
            
            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(f"  ✓ Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]")
        except Exception as e:
            print(f"  ✗ Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  ⚠ GT file NOT found: {gt_path}")

has_gt = gt_loaded_count > 0
print(f"\n{'='*80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics will be {'ENABLED ✓' if has_gt else 'DISABLED (no GT files found)'}")
print(f"{'='*80}\n")


STEP 3: Initialize Comparator & Load Ground Truth
✓ UnifiedModelComparator initialized with 2 models

Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv
  ✓ Loaded GT for Low (Iref=0.28): 758 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv
  ✓ Loaded GT for Medium (Iref=1.0): 758 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv
  ✓ Loaded GT for High (Iref=1.31): 758 points [column: gt_uref_regression]

GT status: 3/3 referen

In [6]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)
rate_tables = []
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]
    print(f"\n▓▓▓ REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref}) ▓▓▓")
    
    # Metrics Table
    df_metrics = comparator.compare_all(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )
    print(f"\nPerformance Metrics:")
    print(df_metrics.to_markdown(index=False))
    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )
    
    # Trend Plot
    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
            uncertainty_style="band",
            uncertainty_opacity=0.12,
            show_series_line=True,
            rate_precision=6,
        )
        print(f"✓ Trend plot saved for {ref_name}")
    except Exception as e:
        print(f"⚠ Trend plot failed for {ref_name}: {e}")
    
    # Detailed Report
    print(f"\nDetailed Comparison Report:")
    comparator.print_comparison_report(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )

STEP 4: Reference-Specific Metrics & Comparison

▓▓▓ REFERENCE CONDITION: Low (Iref=0.28, Tref=57, OHref=10) ▓▓▓

Performance Metrics:
| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I2) |                     0.28 |              5.783 |               468 |        

In [7]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

# Fit quality
try:
    print("\n→ Plotting fit quality (RMSE & R² distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS, output_dir=PLOTS_OUTPUT_DIR)
    print("✓ Fit quality plot saved" if SAVE_PLOTS else "✓ Fit quality plot displayed")
except Exception as e:
    print(f"⚠ Fit quality plot failed: {e}")

# Coefficient diagnostics
try:
    print("\n→ Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(include_c6=True, save=SAVE_PLOTS, output_dir=PLOTS_OUTPUT_DIR)
    print("✓ Coefficient diagnostic plot saved" if SAVE_PLOTS else "✓ Coefficient diagnostic plot displayed")
except Exception as e:
    print(f"⚠ Coefficient diagnostic plot failed: {e}")

# Coverage analysis
try:
    print("\n→ Plotting coverage Gantt (all Irefs)...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS, output_dir=PLOTS_OUTPUT_DIR)
    print("✓ Coverage Gantt saved" if SAVE_PLOTS else "✓ Coverage Gantt displayed")
except Exception as e:
    print(f"⚠ Coverage Gantt failed: {e}")


STEP 5: Cross-Model Diagnostics

→ Plotting fit quality (RMSE & R² distributions)...
✓ Fit quality plot saved

→ Plotting coefficient diagnostics...
✓ Coefficient diagnostic plot saved

→ Plotting coverage Gantt (all Irefs)...
✓ Coverage Gantt saved


In [8]:
# =============================================================================
# 6. CUSTOM REF ANALYSIS (Optional)
# =============================================================================
print("\n" + "=" * 80)
print("STEP 6: Custom Reference Voltage Extraction")
print("=" * 80)

# Build ref_list from REF_CONFIGS
ref_list = []
for ref_name, ref_cfg in REF_CONFIGS.items():
    ref_list.append({
        "Iref": ref_cfg["Iref"],
        "Tref": ref_cfg["Tref"],
        "OHref": ref_cfg["OHref"],
        "name": ref_name,
    })

# For each model, calculate custom Urc values
for model_name, model in models.items():
    print(f"\n→ Calculating custom references for {model_name}...")
    try:
        urc_dict = model.calculate_urc_for_custom_refs(ref_list)
        print(f"✓ Extracted Urc for {len(urc_dict)} reference configurations:")
        for ref_name, urc_df in urc_dict.items():
            print(f"   - {ref_name}: {len(urc_df)} time points")
    except Exception as e:
        print(f"⚠ Failed: {e}")


STEP 6: Custom Reference Voltage Extraction

→ Calculating custom references for Baseline (I2)...
Calculated Urc for Low: 468 points
Calculated Urc for Medium: 468 points
Calculated Urc for High: 468 points
✓ Extracted Urc for 3 reference configurations:
   - Low: 468 time points
   - Medium: 468 time points
   - High: 468 time points

→ Calculating custom references for GPR Smoothed...
Calculated Urc for Low: 762 points
Calculated Urc for Medium: 762 points
Calculated Urc for High: 762 points
✓ Extracted Urc for 3 reference configurations:
   - Low: 762 time points
   - Medium: 762 time points
   - High: 762 time points


In [9]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\n✓ Trained models: {len(models)}")
print(f"✓ Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"✓ Ground truth data: {'Loaded ✓' if has_gt else 'Not available'}")
print(f"✓ Output plots: {PLOTS_OUTPUT_DIR}/" if SAVE_PLOTS else "✓ Plots displayed (not saved)")
print("\nKey settings:")
print(f"  - All condition metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - GT metrics: {SHOW_GT_METRICS}")
print(f"  - Save plots: {SAVE_PLOTS}")


ANALYSIS COMPLETE

✓ Trained models: 2
✓ Reference conditions analyzed: 3
✓ Ground truth data: Loaded ✓
✓ Output plots: ..\\plots\\GPR\\G6M2_comparison/

Key settings:
  - All condition metrics: True
  - GT metrics: True
  - Save plots: True
